In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import joblib
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from src.models.trainer import (
    build_logistic_regression,
    build_random_forest,
    build_linear_svc,
    train_model,
    save_model
)
from src.models.evaluator import (
    evaluate_model,
    print_classification_report,
    plot_confusion_matrix,
    save_metrics_report,
    plot_model_comparison
)
from src.utils.config import MODELS_DIR

print("All imports successful")

In [ ]:
splits = joblib.load(MODELS_DIR / "train_test_splits.pkl")

X_train_tfidf = splits["X_train_tfidf"]
X_test_tfidf  = splits["X_test_tfidf"]
y_cat_train   = splits["y_cat_train"]
y_cat_test    = splits["y_cat_test"]
CATEGORY_LABELS = sorted(y_cat_train.unique().tolist())

print(f"Training matrix: {X_train_tfidf.shape}")
print(f"Test matrix:     {X_test_tfidf.shape}")
print(f"\nClasses: {CATEGORY_LABELS}")

In [ ]:
lr_model = build_logistic_regression()
lr_model = train_model(lr_model, X_train_tfidf, y_cat_train)

In [ ]:
lr_metrics, lr_preds = evaluate_model(
    lr_model, X_test_tfidf, y_cat_test, "Logistic Regression"
)

print_classification_report(y_cat_test, lr_preds, "Logistic Regression")

plot_confusion_matrix(
    y_cat_test, lr_preds,
    "Logistic Regression",
    CATEGORY_LABELS,
    "confusion_matrix_lr.png"
)

In [ ]:
rf_model = build_random_forest()
rf_model = train_model(rf_model, X_train_tfidf, y_cat_train)

In [ ]:
rf_metrics, rf_preds = evaluate_model(
    rf_model, X_test_tfidf, y_cat_test, "Random Forest"
)

print_classification_report(y_cat_test, rf_preds, "Random Forest")

plot_confusion_matrix(
    y_cat_test, rf_preds,
    "Random Forest",
    CATEGORY_LABELS,
    "confusion_matrix_rf.png"
)

In [ ]:
svc_model = build_linear_svc()
svc_model = train_model(svc_model, X_train_tfidf, y_cat_train)

In [ ]:
svc_metrics, svc_preds = evaluate_model(
    svc_model, X_test_tfidf, y_cat_test, "LinearSVC"
)

print_classification_report(y_cat_test, svc_preds, "LinearSVC")

plot_confusion_matrix(
    y_cat_test, svc_preds,
    "LinearSVC",
    CATEGORY_LABELS,
    "confusion_matrix_svc.png"
)

In [ ]:
all_metrics = [lr_metrics, rf_metrics, svc_metrics]

plot_model_comparison(all_metrics, "category_model_comparison.png")

print("\n=== FINAL COMPARISON TABLE ===")
df_results = pd.DataFrame(all_metrics)
df_results = df_results.sort_values("f1_score", ascending=False)
print(df_results.to_string(index=False))

In [ ]:
best = max(all_metrics, key=lambda x: x["f1_score"])
print(f"Best model: {best['model']} with F1={best['f1_score']}")

model_map = {
    "Logistic Regression": lr_model,
    "Random Forest": rf_model,
    "LinearSVC": svc_model,
}

best_model = model_map[best["model"]]
save_model(best_model, "category_classifier.pkl")
save_metrics_report(all_metrics, "category_metrics.json")

print("\nBest model saved as category_classifier.pkl")

In [ ]:
X_test_raw = splits["X_test"]

error_df = pd.DataFrame({
    "text": X_test_raw.values,
    "true": y_cat_test.values,
    "predicted": model_map[best["model"]].predict(X_test_tfidf),
})

errors = error_df[error_df["true"] != error_df["predicted"]]

print(f"Total errors: {len(errors)} out of {len(error_df)} ({len(errors)/len(error_df)*100:.1f}%)")
print("\n=== MOST COMMON CONFUSIONS ===")
confusion_pairs = errors.groupby(["true", "predicted"]).size().sort_values(ascending=False)
print(confusion_pairs.head(10))

print("\n=== SAMPLE MISCLASSIFIED TICKETS ===")
for _, row in errors.head(5).iterrows():
    print(f"\nTrue:      {row['true']}")
    print(f"Predicted: {row['predicted']}")
    print(f"Text:      {str(row['text'])[:200]}")